# ADK Workshop - Environment Verification

## Purpose

This notebook validates your development environment before the workshop. Running these checks **48 hours before the workshop** ensures you catch authentication, dependency, and API access issues early - preventing setup delays during the session.

## Instructions

1. **Run all cells** in order by clicking Runtime → Run all (or Ctrl+F9)
2. **Look for green checkmarks** (✓) indicating successful verification
3. **Fix any failures** using the troubleshooting guidance provided
4. **Confirm success** by seeing "READY FOR WORKSHOP" at the end

**Note:** Complete this verification at least 48 hours before the workshop to allow time for troubleshooting if needed.

In [ ]:
# Cell 2: Project Configuration
# Replace 'your-workshop-project' with your actual GCP project ID

PROJECT_ID = "your-workshop-project"
LOCATION = "us-central1"

print("Configuration:")
print(f"  Project ID: {PROJECT_ID}")
print(f"  Location:   {LOCATION}")
print("\n⚠ Please verify the PROJECT_ID is correct before continuing!")

In [ ]:
# Cell 3: Colab Authentication
# Authenticate with Google Cloud to access Vertex AI

from google.colab import auth
import os

# Authenticate user
auth.authenticate_user(project_id=PROJECT_ID)

# Set environment variables
os.environ['GOOGLE_CLOUD_PROJECT'] = PROJECT_ID
os.environ['GOOGLE_CLOUD_LOCATION'] = LOCATION

print(f"✓ Successfully authenticated with project: {PROJECT_ID}")

In [ ]:
# Cell 4: Install ADK
# Install Google Agent Development Kit version 1.23.0

!pip install -q google-adk==1.23.0

print("✓ google-adk 1.23.0 installed successfully")

In [ ]:
# Cell 5: Verify Python and Dependencies
# Check Python version and ADK installation

import sys
import google.adk
import google.cloud.aiplatform

print("Dependency Verification:")
print("=" * 50)

# Check Python version
python_version = sys.version_info
if python_version >= (3, 11):
    print(f"✓ PASS: Python {python_version.major}.{python_version.minor}.{python_version.micro}")
else:
    print(f"✗ FAIL: Python {python_version.major}.{python_version.minor} (requires 3.11+)")

# Check ADK version
adk_version = google.adk.__version__
if adk_version == '1.23.0':
    print(f"✓ PASS: google-adk {adk_version}")
else:
    print(f"✗ FAIL: google-adk {adk_version} (expected 1.23.0)")

# Verify aiplatform import
print(f"✓ PASS: google.cloud.aiplatform imported successfully")

print("=" * 50)

In [ ]:
# Cell 6: Verify GCP Authentication
# Check that gcloud authentication is active

import subprocess

print("GCP Authentication Status:")
print("=" * 50)

result = subprocess.run(
    ['gcloud', 'auth', 'list'],
    capture_output=True,
    text=True
)

if result.returncode == 0 and 'ACTIVE' in result.stdout:
    print("✓ PASS: GCP authentication active")
    # Show active account
    for line in result.stdout.split('\n'):
        if 'ACTIVE' in line:
            print(f"  Account: {line.strip()}")
            break
else:
    print("✗ FAIL: No active GCP authentication")
    print("  Run the authentication cell above")

print("=" * 50)

In [ ]:
# Cell 7: Verify Vertex AI API Access
# Confirm Vertex AI API is enabled and accessible

from google.cloud import aiplatform

print("Vertex AI API Access:")
print("=" * 50)

try:
    aiplatform.init(project=PROJECT_ID, location=LOCATION)
    print(f"✓ PASS: Vertex AI API accessible")
    print(f"  Project: {PROJECT_ID}")
    print(f"  Location: {LOCATION}")
except Exception as e:
    print(f"✗ FAIL: Vertex AI API not accessible")
    print(f"  Error: {str(e)}")

print("=" * 50)

In [ ]:
# Cell 8: Test Gemini Model Access
# Create a test agent and verify Gemini 2.5 Flash responds

from google.adk.agents import Agent

print("Gemini Model Access Test:")
print("=" * 50)

try:
    # Create minimal test agent
    test_agent = Agent(
        model='gemini-2.5-flash',
        name='test_agent',
        description='Environment verification test agent.',
        instruction='Respond with a brief greeting.',
    )
    
    # Call the agent
    response = test_agent.generate_content("Hello")
    
    # Verify response exists
    if response.text and len(response.text) > 0:
        print("✓ PASS: Gemini 2.5 Flash model responding")
        print(f"  Model response preview: {response.text[:100]}...")
    else:
        print("✗ FAIL: Empty response from Gemini model")
        
except Exception as e:
    print(f"✗ FAIL: Cannot access Gemini model")
    print(f"  Error: {str(e)}")

print("=" * 50)

In [ ]:
# Cell 9: Summary Verification Function
# Consolidated verification report

def verify_environment():
    """
    Run all environment checks and return consolidated pass/fail report.
    Returns True if all checks pass, False otherwise.
    """
    import sys
    import subprocess
    import google.adk
    from google.cloud import aiplatform
    from google.adk.agents import Agent
    
    checks = {}
    
    # Check 1: Python version
    python_version = sys.version_info
    checks['Python 3.11+'] = python_version >= (3, 11)
    
    # Check 2: ADK version
    checks['google-adk 1.23.0'] = google.adk.__version__ == '1.23.0'
    
    # Check 3: GCP authentication
    try:
        result = subprocess.run(['gcloud', 'auth', 'list'],
                              capture_output=True, text=True, timeout=5)
        checks['GCP authenticated'] = 'ACTIVE' in result.stdout
    except:
        checks['GCP authenticated'] = False
    
    # Check 4: Vertex AI API
    try:
        aiplatform.init(project=PROJECT_ID, location=LOCATION)
        checks['Vertex AI API enabled'] = True
    except:
        checks['Vertex AI API enabled'] = False
    
    # Check 5: Gemini model access
    try:
        test_agent = Agent(
            model='gemini-2.5-flash',
            name='verify',
            description='Test',
            instruction='Say hi',
        )
        response = test_agent.generate_content("Test")
        checks['Gemini 2.5 Flash access'] = len(response.text) > 0
    except:
        checks['Gemini 2.5 Flash access'] = False
    
    # Print consolidated report
    print("\n" + "=" * 60)
    print("ENVIRONMENT VERIFICATION SUMMARY")
    print("=" * 60)
    
    passed = 0
    failed = 0
    
    for check_name, result in checks.items():
        if result:
            print(f"✓ PASS: {check_name}")
            passed += 1
        else:
            print(f"✗ FAIL: {check_name}")
            failed += 1
    
    print("=" * 60)
    print(f"Results: {passed} passed, {failed} failed")
    print("=" * 60)
    
    all_passed = failed == 0
    
    if all_passed:
        print("\n🎉 READY FOR WORKSHOP!")
        print("All checks passed. You're good to go!")
    else:
        print("\n⚠ NOT READY - TROUBLESHOOTING NEEDED")
        print("Please fix the failed checks above.")
        print("Troubleshooting guide: [Workshop troubleshooting URL]")
    
    return all_passed

# Run verification
verify_environment()

## Next Steps

### ✓ All Checks Passed

**Congratulations! You're ready for the workshop!**

Your environment is properly configured with:
- Python 3.11+ installed
- Google ADK 1.23.0 installed
- GCP authentication working
- Vertex AI API enabled and accessible
- Gemini 2.5 Flash model responding

See you at the workshop!

### ✗ Some Checks Failed

**Don't worry! Follow these steps:**

1. Review the error messages above for specific failures
2. Consult the troubleshooting guide: [Workshop troubleshooting URL]
3. Contact the workshop organizer if you need help:
   - Email: [workshop-support@example.com]
   - Slack: [#workshop-help channel]

**Please resolve issues at least 24 hours before the workshop to ensure a smooth experience.**